# Outlook

In this notebook we code the clipped version of the [Proximal Policy
Optimization (PPO)](https://arxiv.org/pdf/1707.06347.pdf) algorithm.

The PPO algorithm is explained in [this
video](https://www.youtube.com/watch?v=uRNL93jV2HE) and you can also read
[the corresponding
slides](http://pages.isir.upmc.fr/~sigaud/teach/ps/10_ppo.pdf).

It is also a good idea to have a look at the [spinning up
documentation](https://spinningup.openai.com/en/latest/algorithms/ppo.html)
and, for the many details that make PPO work in practice, at [The 37
implementation details of
PPO](https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/).

## Setup

**On Colab**: run the installation cell below (generated automatically,
pinned versions). **Locally**: install the environment with `uv sync`, no
need for this cell.

In [9]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, replace
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.profiles import Profile
from rl_mind.core import Actor, StochasticAction, TensorStruct
from rl_mind.nn import build_mlp
from rl_mind.env import VecEnv
from rl_mind.data import Rollout, minibatches
from rl_mind.collectors import RolloutCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings
from copy import deepcopy

## The `rl_mind` toolkit

The `rl_mind` package ([documentation](https://master-dac.isir.upmc.fr/rl/rl-mind/)),
installed with the practicals, provides a small set of *typed* helpers used
throughout the RL notebooks. For this notebook, the important ones are:

- `VecEnv` runs several copies of a gymnasium environment in parallel,
  using torch tensors;
- `Rollout` is a fixed-length segment of interaction with `B` parallel
  environments: all its tensors have shape `[T, B, ...]`, and
  `next_obs[t]` is always the true successor of `obs[t]` (the *final*
  observation when an episode ends at step `t`);
- `RolloutCollector` runs an actor in a `VecEnv` and returns `Rollout`s;
- `minibatches` iterates over random minibatches of a batch of data, using
  each sample exactly once;
- `Evaluator` periodically evaluates an actor and keeps a copy of the best
  one.

In [2]:
# pygame and tensorboard still use the deprecated `pkg_resources` API
silence_known_warnings()

# PPO

PPO is an *on-policy* policy-gradient algorithm: it collects a fixed-size
rollout with the current policy, then performs **several epochs** of
gradient updates on this data. Reusing the same samples several times
would normally push the policy too far from the one that collected the
data; PPO-clip prevents this by *clipping* the policy improvement.

More precisely, PPO-clip maximizes:

$$
L^{CLIP}(\theta)=\hat{\mathbb{E}}_t\left[\min \left(r_t(\theta)
\hat{A}_t,\ \operatorname{clip}\left(r_t(\theta), 1-\epsilon,
1+\epsilon\right) \hat{A}_t\right)\right]
$$

where $\hat{A}_t$ is the advantage and

$$r_t(\theta) = \frac{\pi_\theta\left(a_t \mid
s_t\right)}{\pi_{\theta_{\text{old}}}\left(a_t \mid s_t\right)}$$

is the probability ratio between the current policy and the policy that
collected the data. Note that we do not need to keep a copy of the old
policy: it is enough to have stored the log-probabilities of the actions
when they were sampled — which is exactly what the `StochasticAction`
returned by the policy contains.

## Definition of the agents

### The policy

The policy maps a state to a **distribution** over the (discrete)
actions. During training (`forward`), the action is *sampled* and its
log-probability is stored in the returned `StochasticAction`; during
evaluation (`act`), we take the most probable action.

In [3]:
class DiscretePolicy(Actor[StochasticAction]):
    """A stochastic policy for discrete actions, $\\pi(a|s)$"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...], n_actions: int):
        super().__init__()
        self.model = build_mlp([obs_dim, *hidden, n_actions])

    def dist(self, obs: Tensor) -> torch.distributions.Categorical:
        """The distribution over actions for a batch of observations"""
        return torch.distributions.Categorical(logits=self.model(obs))

    def forward(self, obs: Tensor) -> StochasticAction:
        dist = self.dist(obs)
        action = dist.sample()
        return StochasticAction(value=action, log_prob=dist.log_prob(action))

    def act(self, obs: Tensor) -> Tensor:
        return self.model(obs).argmax(-1)

### The critic $V(s)$

As in A2C, PPO uses a value function $V(s)$, learned by regression
towards the observed returns; it is used to compute the advantages.

In [4]:
class ValueNetwork(nn.Module):
    """The value network $V(s)$"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...]):
        super().__init__()
        self.model = build_mlp([obs_dim, *hidden, 1])

    def forward(self, obs: Tensor) -> Tensor:
        """Compute $V(s)$ for a batch: `[..., obs_dim] -> [...]`"""
        return self.model(obs).squeeze(-1)

## Generalized Advantage Estimation (GAE)

The advantage $\hat{A}_t$ is estimated with
[GAE](https://arxiv.org/abs/1506.02438), a geometric average of $n$-step
advantage estimators controlled by $\lambda$ ($\lambda = 0$ gives the
one-step temporal difference, $\lambda = 1$ the Monte-Carlo estimate):

$$\hat{A}_t = \delta_t + \gamma \lambda\, \hat{A}_{t+1}, \qquad
\delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)$$

The recursion is computed backwards over the rollout. Two subtleties at
episode boundaries (this is where `terminated` / `truncated` matter):

- when $s_{t+1}$ is a **terminal** state, $V(s_{t+1})$ must not be used in
  $\delta_t$ (no bootstrap);
- the recursion must not propagate across episode boundaries (`done`),
  whether terminated or truncated.

Write the backward recursion below. `rollout.terminated[t]` and
`rollout.done[t]` are boolean tensors of shape `[B]`, so `(~x).float()` is
the mask that zeroes out the corresponding term.

In [8]:
def compute_gae(
    gamma: float, lam: float, rollout: Rollout, values: Tensor, next_values: Tensor
) -> Tensor:
    """Compute the GAE advantages over a rollout

    :param gamma: The discount factor
    :param lam: The GAE $\\lambda$ parameter
    :param rollout: The rollout (tensors `[T, B]`)
    :param values: $V(s_t)$, shape `[T, B]`
    :param next_values: $V(s_{t+1})$, shape `[T, B]`
    :return: The advantages, shape `[T, B]`
    """
    # Compute the GAE advantages (backward recursion over time)

    advantages = torch.zeros_like(rollout.reward)
    # The running advantage, one value per environment
    gae = torch.zeros_like(rollout.reward[0])
    for t in reversed(range(rollout.reward.shape[0])):
        # the TD error, without bootstrap on terminal states
        delta = rollout.reward[t] + gamma * next_values * (~rollout.terminated[t]).float() - values
        # accumulate, but not across episode boundaries
        gae = delta + (gamma * lam * gae) * (~rollout.done[t]).float()
        advantages[t] = gae
    return advantages



## The training data

After collecting a rollout, we compute the advantages and flatten
everything into one batch of `T*B` samples: this is exactly the data PPO
trains on during its inner optimization epochs. The `returns` field
($\hat{A}_t + V(s_t)$) is the regression target of the critic.

In [6]:
@dataclass(frozen=True)
class PPOBatch(TensorStruct):
    """The data PPO trains on (all shapes `[N, ...]`)

    :param obs: The observations
    :param action: The actions that were played
    :param old_log_prob: The log-probabilities of these actions *when they
        were sampled* (i.e. under the old policy)
    :param advantage: The GAE advantages
    :param returns: The regression targets for the critic
    """

    obs: Tensor
    action: Tensor
    old_log_prob: Tensor
    advantage: Tensor
    returns: Tensor

## Hyper-parameters

In [7]:
@dataclass(frozen=True)
class PPOClipConfig:
    env_name: str = "CartPole-v1"
    seed: int = 1

    #: Number of rollouts (updates)
    max_updates: int = Profile.pick(100, fast_test=3)
    #: Number of parallel training environments
    n_envs: int = 8
    #: Rollout length (per environment)
    n_steps: int = 128

    #: Discount factor
    gamma: float = 0.99
    #: GAE lambda
    gae_lam: float = 0.95

    #: PPO clipping parameter $\epsilon$
    epsilon: float = 0.2
    #: Maximum number of optimization epochs over each rollout
    opt_epochs: int = 10
    #: Stop the epochs early when the KL to the old policy exceeds this
    #: (the "trust region"); `None` disables the check
    target_kl: float | None = 0.02
    #: Minibatch size
    batch_size: int = 64
    #: Normalize the advantages in each minibatch
    normalize_advantage: bool = True
    #: Entropy bonus coefficient
    entropy_coef: float = 0.0

    actor_hidden: tuple[int, ...] = (64, 64)
    critic_hidden: tuple[int, ...] = (64, 64)
    lr: float = 1e-3

    #: Steps between two evaluations, and number of evaluation episodes
    eval_interval: int = Profile.pick(8_000, fast_test=1_000)
    n_eval_envs: int = 10

## Main training loop

Note the structure: collect a rollout (with the *current* policy), compute
the advantages **once** (no gradient), then run several epochs of
minibatch updates on this frozen data. `minibatches` partitions the data
randomly, using each sample exactly once per epoch.

**How many epochs?** Each epoch moves $\pi_\theta$ further away from the
$\pi_{\theta_{\text{old}}}$ that collected the data, and the objective is
only a valid local approximation while the two stay close — the *trust
region*. Clipping bounds the *per-sample* ratio, but nothing stops the
policy as a whole from drifting after several epochs. So on top of the
`opt_epochs` limit, we monitor the KL divergence to the old policy and stop
as soon as it exceeds `cfg.target_kl`. The estimator used below,

$$\widehat{\operatorname{KL}} = \hat{\mathbb{E}}_t\big[(r_t(\theta) - 1) -
\log r_t(\theta)\big]$$

is unbiased, always positive, and has a much lower variance than the naive
$-\hat{\mathbb{E}}_t[\log r_t(\theta)]$.

You have two things to write in the loop below: the **PPO-clip policy
loss**, and the **trust-region stopping criterion**. Useful torch
functions:

- [`torch.clamp`](https://pytorch.org/docs/stable/generated/torch.clamp.html)
  computes $\min(\max(x, m), M)$;
- [`torch.minimum`](https://pytorch.org/docs/stable/generated/torch.minimum.html)
  computes the element-wise minimum of two tensors;
- `dist.log_prob(actions)` and `dist.entropy()` where `dist =
  policy.dist(obs)`.

Remember that the ratio $r_t(\theta)$ is more conveniently computed from
the *difference of log-probabilities* (keep that difference around — the KL
estimator needs it too), and that we *minimize* the negated objective.

In [ ]:
def run_ppo_clip(cfg: PPOClipConfig) -> Evaluator:
    torch.manual_seed(cfg.seed)
    # same_step_reset: every row of the rollout is a valid transition
    env = VecEnv(cfg.env_name, cfg.n_envs, seed=cfg.seed, same_step_reset=True)

    policy = DiscretePolicy(env.observation_dim, cfg.actor_hidden, env.n_actions)
    critic = ValueNetwork(env.observation_dim, cfg.critic_hidden)
    policy_optimizer = torch.optim.Adam(policy.parameters(), lr=cfg.lr)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=cfg.lr)

    collector = RolloutCollector(env, policy)
    run_dir = run_directory(f"ppo-clip-{cfg.env_name}-S{cfg.seed}")
    writer = SummaryWriter(run_dir)
    evaluator = Evaluator(
        VecEnv(cfg.env_name, cfg.n_eval_envs, seed=cfg.seed + 100),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=writer,
    )

    pbar = tqdm(range(cfg.max_updates))
    for update in pbar:
        rollout = collector.collect(cfg.n_steps)

        # Compute the advantages and returns (once per rollout, no gradient)
        with torch.no_grad():
            values = critic(rollout.obs)
            next_values = critic(rollout.next_obs)
            advantage = compute_gae(
                cfg.gamma, cfg.gae_lam, rollout, values, next_values
            )

        flat = rollout.flatten()
        data = PPOBatch(
            obs=flat.obs,
            action=flat.action.value,
            old_log_prob=flat.action.log_prob,
            advantage=advantage.flatten(),
            returns=(advantage + values).flatten(),
        )

        # Inner optimization loop: several epochs over the same rollout, for
        # as long as the policy stays within the trust region
        epochs_done = 0

        old_policy = deepcopy(policy)
        for epoch in range(cfg.opt_epochs):
            epoch_kl = []
            for mb in minibatches(data, cfg.batch_size):
                # Compute the PPO-clip policy loss (and the entropy bonus)

                dist = policy.dist(mb.obs)
                with torch.no_grad:
                    old_dist = old_policy.dist(mb.obs)
                # The log-ratio, then the probability ratio, between the
                # current policy and the policy that collected the data
                log_ratio = dist.log_prob(mb.action) - old_dist.log_prob(mb.action)
                ratio = torch.exp(log_ratio)
                advantage_mb = mb.advantage
                if cfg.normalize_advantage:
                    advantage_mb = (advantage_mb - advantage_mb.mean()) / (
                        advantage_mb.std() + 1e-8
                    )
                policy_loss = torch.minimum()
                loss = policy_loss - cfg.entropy_coef * dist.entropy().mean()
                assert False, 'Not implemented yet'


                policy_optimizer.zero_grad()
                loss.backward()
                policy_optimizer.step()

                # Critic update: regression towards the returns
                critic_loss = F.mse_loss(critic(mb.obs), mb.returns)
                critic_optimizer.zero_grad()
                critic_loss.backward()
                critic_optimizer.step()

                # How far the policy has moved, on this minibatch
                with torch.no_grad():
                    epoch_kl.append(((ratio - 1) - log_ratio).mean().item())

            # Stop the epochs when the policy left the trust region

            # `epoch_kl` holds one KL estimate per minibatch: compare their
            # mean to `cfg.target_kl` (when it is not None) and `break`
            assert False, 'Not implemented yet'


        writer.add_scalar("train/epochs", epochs_done, collector.steps)
        writer.add_scalar("loss/policy", policy_loss.item(), collector.steps)
        writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)
        if result := evaluator.run_if_needed(collector.steps, policy):
            pbar.set_description(
                f"eval={result.mean:6.1f} best={evaluator.best_reward:6.1f}"
            )

    return evaluator

## Launching tensorboard to visualize the results

In [ ]:
setup_tensorboard()

In [ ]:
ppo_config = PPOClipConfig()
ppo_clip = run_ppo_clip(ppo_config)

In [ ]:
ppo_clip.visualize_best()

# Learning curve

In [ ]:
steps = [result.step for result in ppo_clip.history]
means = torch.tensor([result.mean for result in ppo_clip.history])
stds = torch.tensor([float(result.rewards.std()) for result in ppo_clip.history])
plt.plot(steps, means)
plt.fill_between(steps, means - stds, means + stds, alpha=0.2)
plt.xlabel("steps")
plt.ylabel("cumulated reward")
plt.title("PPO-clip on CartPole-v1")
plt.show()

## Exercises

- Set `epsilon` to a large value (e.g. 100, i.e. no clipping) and
  compare the learning curves: you should observe the instability that
  clipping prevents.
- Compare `opt_epochs=1` with `opt_epochs=10`: the whole point of PPO is
  to be able to reuse the collected data several times.
- Play with the trust region: `target_kl=None` (no early stopping),
  `target_kl=0.01` (very conservative). Look at the `train/epochs` curve in
  tensorboard to see how many epochs actually run, and relate it to the
  learning curve. With `opt_epochs=1` the criterion never triggers — that
  configuration is essentially A2C with a clipped objective.
- Try a harder environment (e.g. `LunarLander-v3`), or a continuous-action
  environment by replacing the `DiscretePolicy` with a Gaussian policy
  returning a `StochasticAction`.